# Ray Serve LLM: the framework and how to deploy a dense model

© 2026, Anyscale. All Rights Reserved

Serving an LLM is not just serving a bigger model: most of what classic online inference assumes stops holding. This notebook starts with what changes, then builds a real OpenAI-compatible endpoint with Ray Serve LLM and shows how the same config scales a dense model across GPUs.

<div class="alert alert-block alert-info">
<b>Roadmap for this notebook</b>
<ol>
    <li>Why LLM inference needs its own engine.</li>
    <li>Why choose Ray Serve LLM.</li>
    <li>Build and run your first LLM with Ray Serve LLM.</li>
    <li>Scaling on the framework and engine levels.</li>
    <li>Scaling the fleet: autoscaling and multi-model.</li>
    <li>Composing the LLM with other deployments.</li>
    <li>Deploying a production-ready LLM.</li>
</ol>
</div>

**Imports**

In [ ]:
from openai import OpenAI
from ray import serve
from ray.serve.llm import LLMConfig, build_openai_app

## 1. Why LLM inference needs its own engine

### 1.1 What changes from classic inference

A classic online model — a ranker, a classifier, an embedding model — behaves like a function: one input, one forward pass, one output. An LLM breaks that in four ways.

<img src="https://anyscale-public-materials.s3.us-west-2.amazonaws.com/ray-serve-distributed-inference/diagrams/llm_vs_classic_rev3.png" loading="lazy" width="830">

What each difference forces:

- **Work per request**
    - The output length is unknown until the model stops, so a request cannot be sized, predicted, or scheduled as a unit.
- **Batching**
    - Prompt and output lengths both vary, so a fixed batch idles as sequences finish early.
- **Capacity**
    - The weights are fixed, but a sequence's **key-value (KV) cache** grows every token and can outgrow them.
    - So how many requests fit is a runtime question, set by context length.
- **State**
    - That cache lives on the replica that built it, so replicas stop being interchangeable.

- **The first row causes the other three.** Once output length is unknown, the batch cannot be fixed, the memory cannot be computed ahead, and the cache keeps growing on whichever replica started the request.
- **So LLM serving grew a specialized engine** rather than a bigger model server. Section 1.2 names that layer and the one above it.

### 1.2 Engines and frameworks

An **engine** makes a single replica fast; a **framework** coordinates many replicas. They are different jobs, and most teams need both.

- **The engine owns one replica.** Paged KV cache, continuous batching, fused kernels, and intra-replica tensor and pipeline parallelism all live here. vLLM and SGLang are engines. The engine is where tokens are generated.
- **The framework owns the fleet.** Routing across replicas, autoscaling, prefill/decode disaggregation orchestration, multi-model composition, and the public API surface. A framework does not generate tokens; it decides which engine instance does, and how many exist.
- **Ray Serve LLM is a framework that runs the same engines everyone else runs** (vLLM by default, SGLang optional). So a framework comparison is never "whose kernels are faster" (the kernels are shared), but "whose fleet coordination and developer surface fit your problem."

The two layers, and the seam between them where a framework manages N engine replicas:

<img src="https://anyscale-public-materials.s3.us-west-2.amazonaws.com/ray-serve-distributed-inference/diagrams/engine_vs_framework.png" loading="lazy" width="760">

`LLMConfig` is the framework's description of one model deployment. Constructing one runs no GPU work, so you can see which side of that seam it sits on before serving anything.

In [ ]:
preview = LLMConfig(
    model_loading_config=dict(
        model_id="qwen-0.5b",
        model_source="s3://anyscale-public-materials-use2/models/Qwen/Qwen2.5-0.5B-Instruct",
    ),
)
type(preview).__module__, type(preview).__name__   # it is a ray.serve.llm class

Note: `LLMConfig` is a `ray.serve.llm` object, part of the framework. The engine (vLLM) is configured *through* it, not imported directly.

## 2. Why choose Ray Serve LLM

Every framework in this layer runs the same engines, so the choice is about fleet coordination and developer surface, not token speed. Ray Serve LLM stakes out four genuine differentiators.

<img src="https://anyscale-public-materials.s3.us-west-2.amazonaws.com/ray-serve-distributed-inference/diagrams/why_ray_serve_llm.png" loading="lazy" width="1000">

- **Python-first control plane.** Build and ship from a `.py` with `build_openai_app`: routing, composition, and autoscaling can be expressed in Python, not restricted to YAML. Serve takes a config YAML too, so the control plane is a choice rather than a constraint.
- **General compute, not LLM-only.** The LLM is one deployment in an ordinary Serve app, so the same app can hold non-LLM models (a ranker, a featurizer) and arbitrary Python, composed with `.bind()`.
- **Accelerator-agnostic.** One field, `LLMConfig.accelerator_type`, spans NVIDIA, AMD Instinct, Intel Gaudi and GPU-Max, AWS Neuron, Google TPU, and Huawei Ascend.
- **Runs the same engines and KV transports as the alternatives.** vLLM by default, SGLang optional, so one Ray Serve LLM replica is exactly as fast as the vLLM or SGLang it wraps. The differences are in coordination and surface, not token speed.

Teams standardizing on a Kubernetes or Gateway-class serving substrate have mature options there too; Ray Serve LLM is the choice when you want a Python-first control plane and general compute on Ray. With that placed, the rest of the notebook builds.

## 3. Build and run your first LLM with Ray Serve LLM

The whole framework surface is one config object, one builder, and the stock OpenAI client. This section configures the model, builds the app, runs it on the GPU, and queries it.

### 3.1 Anatomy of an `LLMConfig`

`LLMConfig` is the one object you fill in. Its four blocks answer four questions: *what* model, *which* accelerator, *how many* replicas, and *how* the engine runs. Color-coded to where each block lands at runtime:

<img src="https://anyscale-public-materials.s3.us-west-2.amazonaws.com/ray-serve-distributed-inference/diagrams/llmconfig_anatomy.png" loading="lazy" width="760">

In [ ]:
BUCKET_URI = "s3://anyscale-public-materials-use2/models/Qwen/Qwen2.5-0.5B-Instruct"

llm_config = LLMConfig(
    model_loading_config=dict(
        model_id="qwen-0.5b",       # how clients address it (the OpenAI model= field)
        model_source=BUCKET_URI,    # stream weights from the bucket, not the HF Hub
    ),
    # accelerator_type omitted: one GPU type in this fleet, so Serve uses the only accelerator present
    deployment_config=dict(
        autoscaling_config=dict(min_replicas=1, max_replicas=2),   # Serve scales whole replicas
    ),
    runtime_env=dict(env_vars={"AWS_REGION": "us-east-2"}),   # the region this bucket lives in
    engine_kwargs=dict(
        load_format="runai_streamer",  # vLLM streams safetensors from S3 into GPU memory
        max_model_len=4096,            # cap the context to keep the KV cache small and cold start fast
        enforce_eager=True,            # skip CUDA-graph capture for a faster cold start
    ),
)
llm_config.model_loading_config.model_id

<div class="alert alert-block alert-info">
<b><code>accelerator_type</code> is optional.</b> In a homogeneous fleet (every node the same GPU) you can omit it and Serve schedules on the only accelerator there is. Its real job is a <i>heterogeneous</i> fleet under one Ray cluster: it pins each model to a specific accelerator, so you serve several models on different hardware at once (a 0.5B on L4, a 70B on H100) from one cluster.
</div>

### 3.2 Build the app and run it

One builder turns that config into an OpenAI-compatible ingress in front of one engine replica.

<img src="https://anyscale-public-materials.s3.us-west-2.amazonaws.com/ray-serve-distributed-inference/diagrams/llm_app_shape.png" loading="lazy" width="820">

`build_openai_app` takes a dict with a list of configs and returns a ready `Application`: an OpenAI-compatible ingress in front of one `LLMServer` replica per config.

In [ ]:
app = build_openai_app({"llm_configs": [llm_config]})
serve.run(app, blocking=False)

Note: the first run streams the weights from the bucket onto the GPU, so the replica takes a moment to become healthy. Once `serve.run` returns, the endpoint is live at `localhost:8000`. Passing several configs serves several models behind one ingress (section 5).

### 3.3 Query it: chat, then streaming

Because the app speaks the OpenAI REST API, any OpenAI client works. Point `base_url` at the ingress and use any non-empty `api_key` (Serve does not check it locally).

In [ ]:
client = OpenAI(base_url="http://localhost:8000/v1", api_key="fake-key")
resp = client.chat.completions.create(
    model="qwen-0.5b",
    messages=[{"role": "user", "content": "In one sentence, what does an inference framework do?"}],
    max_tokens=128,
)
print(resp.choices[0].message.content)

Now enable streaming with `stream=True`, so tokens arrive as fast as they are generated instead of after the whole reply is complete.

In [ ]:
stream = client.chat.completions.create(
    model="qwen-0.5b",
    messages=[{"role": "user", "content": "Count from one to five."}],
    max_tokens=64,
    stream=True,
)
for chunk in stream:
    if chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)
print()

Tear down the app before the concepts in section 4. The rest of the notebook is display blocks, so this is the last running cell.

In [ ]:
serve.shutdown()

## 4. Scaling on the framework and engine levels

A 0.5B model fits one GPU, so the section-3 app used one GPU per replica. A replica is really a *group* of GPUs, and that one shape carries two scaling levers you must hold at once, not merge: the **framework replicates whole replicas**, and **one replica's GPUs run a form of parallelism** when a model is too big for a single GPU.

### 4.1 Two independent levers

- **Lever 1: the framework replicates whole replicas.** Autoscaling adds and removes complete replicas; each replica is an independent copy of the model serving its share of traffic. This is how the *fleet* scales (section 5).
- **Lever 2: a replica's GPUs run a form of parallelism.** When a model is too big for one GPU, the replica's GPUs split the work with tensor (TP), pipeline (PP), data (DP), or expert (EP) parallel, often combined. This is how *one replica* scales.
- **Engine vs replica scope.** The engine scales the work *inside* one replica (batching, the KV cache, the per-layer math across its GPUs); the framework scales the *number* of replicas. Different layers, different levers.
- **A replica reserves its GPUs as a unit.** You set the parallel sizes and Serve does the GPU bookkeeping.
- **The framework layer is always Ray.** Whether the engine's internal workers use Ray is the engine's business, not something you configure.

The ingress fans into whole replicas (the framework's lever); each replica is itself a group of GPUs sliced by tensor, pipeline, data, or expert parallel (the replica's lever):

<img src="https://anyscale-public-materials.s3.us-west-2.amazonaws.com/ray-serve-distributed-inference/diagrams/nb3_scaling_axes.png" loading="lazy" width="960">

### 4.2 Tensor parallel vs pipeline parallel: two ways to slice one replica

Two ways to cut one replica across its GPUs: shard every layer, or split the layer stack.

<img src="https://anyscale-public-materials.s3.us-west-2.amazonaws.com/ray-serve-distributed-inference/diagrams/tp_vs_pp_rev1.png" loading="lazy" width="900">

When one replica must span GPUs, two `engine_kwargs` integers decide how it is sliced:

- **`tensor_parallel_size` (tensor parallel).** Shard each layer's weights across N GPUs. Many small all-reduces every layer, so it wants fast intra-node links (NVLink). This is the default lever for "the model is too big for one GPU."
- **`pipeline_parallel_size` (pipeline parallel).** Split the layer stack into contiguous stages, one hand-off per stage boundary. It tolerates slower links, so it is the across-node lever.
- Both are just `engine_kwargs` integers; Serve reserves the matching GPUs for the replica as a unit.

The dials that slice one replica across four GPUs:

```python
# A replica sliced across 4 GPUs: shard each layer (TP) and split the layer stack (PP).
big_replica = LLMConfig(
    model_loading_config=dict(model_id="demo", model_source="some/large-model"),
    accelerator_type="H100",
    engine_kwargs=dict(tensor_parallel_size=2, pipeline_parallel_size=2),
)
```

## 5. Scaling the fleet: autoscaling and multi-model

Section 4's first lever, replicating whole replicas, becomes two concrete framework knobs here: autoscaling the replica count, then several models behind one ingress.

### 5.1 Autoscaling LLM replicas

The same control loop any Serve deployment uses, with a much longer cold start because each replica loads weights onto a GPU.

<img src="https://anyscale-public-materials.s3.us-west-2.amazonaws.com/ray-serve-distributed-inference/diagrams/autoscaling_llm.png" loading="lazy" width="900">

Autoscaling lives under `deployment_config.autoscaling_config`, the same Serve keys any deployment uses:

```python
llm_config = LLMConfig(
    model_loading_config=dict(model_id="qwen-0.5b", model_source=BUCKET_URI),
    runtime_env=dict(env_vars={"AWS_REGION": "us-east-2"}),
    engine_kwargs=dict(load_format="runai_streamer", max_model_len=4096, enforce_eager=True),
    deployment_config=dict(autoscaling_config=dict(
        min_replicas=1,              # the warm pool: never pay a cold start on the first request
        initial_replicas=2,          # start here, then let the loop take over
        max_replicas=4,
        target_ongoing_requests=8,   # a GPU replica with continuous batching holds many in flight
        upscale_delay_s=10,          # default 30; react sooner because a replica takes minutes
        upscaling_factor=2.0,        # overshoot the computed step, to scale out in fewer rounds
        downscale_delay_s=600,       # downscale lazily so a token-streaming spike doesn't thrash
    )),
)
```

An LLM replica holds a GPU plus a paged KV cache, so cold start is minutes, not seconds. Both directions therefore want tuning away from the defaults, in opposite ways.

| knob | default | why an LLM wants it different |
|---|---:|---|
| `min_replicas` | 1 | the warm pool floor; at 0, the first request after idle waits out a full load |
| `initial_replicas` | unset | start warm at a known level rather than climbing from `min_replicas` |
| `upscale_delay_s` | 30 | lower it: the delay is spent *before* a replica that itself takes minutes to arrive |
| `upscaling_factor` | 1.0 | raise it above 1 to overshoot each step and reach capacity in fewer rounds |
| `downscale_delay_s` | 600 | keep it long: releasing a GPU you re-acquire minutes later is the expensive mistake |

- **The factor is a gain on the step, not on the target**
    - The loop computes `current + factor * (desired - current)`.
    - At `1.0` it goes straight to the computed target; above `1.0` it overshoots; below, it creeps.
- **Scaling from zero ignores the factor entirely**
    - At 0 replicas the whole desired count would be treated as the delta and amplified every tick, so the loop bypasses the factor.
    - Another reason to hold a warm floor rather than tune around a cold one.
- **`downscale_to_zero_delay_s` splits the last step**
    - It governs only the 1 to 0 transition, and falls back to `downscale_delay_s` when unset.
    - Set it far higher to keep one replica warm long after the rest have gone.

### 5.2 Multi-model behind one ingress

One endpoint, many models, each scaling independently.

<img src="https://anyscale-public-materials.s3.us-west-2.amazonaws.com/ray-serve-distributed-inference/diagrams/multi_model_ingress.png" loading="lazy" width="860">

Pass a list of `LLMConfig`s and the one ingress dispatches by the client's `model=` field:

```python
configs = [
    LLMConfig(model_loading_config=dict(model_id="qwen-0.5b", model_source="Qwen/Qwen2.5-0.5B-Instruct"),
              engine_kwargs=dict(max_model_len=4096, enforce_eager=True)),
    LLMConfig(model_loading_config=dict(model_id="qwen-1.5b", model_source="Qwen/Qwen2.5-1.5B-Instruct"),
              engine_kwargs=dict(max_model_len=4096, enforce_eager=True)),
]
serve.run(build_openai_app({"llm_configs": configs}), blocking=False)
# client.chat.completions.create(model="qwen-0.5b", ...) vs model="qwen-1.5b"
```

Note: each config is its own deployment with its own replicas, autoscaler, and router. This is the seam for A/B tests, tiered models, or a cheap-then-escalate pattern, all in one Python app. These two use Hugging Face ids rather than the `s3://` mirror from section 3.1, since only the 0.5B is mirrored.

## 6. Composing the LLM with other deployments

`build_openai_app` hands you a finished service. When the LLM should instead be one part of a larger application, drop one level: `build_llm_deployment` returns the `LLMServer` deployment on its own.

- **It is an ordinary Serve deployment**
    - `build_llm_deployment(llm_config)` and `Triage.bind()` both return an `Application`.
    - So the ingress binds them the same way and cannot tell them apart.
- **The routing logic is a Python `if`**
    - A CPU deployment answers what it can; only the rest reaches the GPU.
- **One application, one lifecycle**
    - The GPU deployment and the CPU deployment scale and deploy together.

```python
from ray.serve.handle import DeploymentHandle
from ray.serve.llm import build_llm_deployment
from ray.serve.llm.openai_api_models import ChatCompletionRequest

@serve.deployment(ray_actor_options={"num_cpus": 1})
class Triage:                                  # a plain deployment: no GPU, no LLM
    async def route(self, text: str) -> str | None:
        return next((k for k in FAQ if k in text.lower()), None)

@serve.deployment
@serve.ingress(api)
class Assistant:
    def __init__(self, triage: DeploymentHandle, llm: DeploymentHandle) -> None:
        self.triage = triage
        self.llm = llm.options(stream=True)    # LLMServer.chat is an async generator

    @api.post("/ask")
    async def ask(self, body: Ask) -> dict:
        hit = await self.triage.route.remote(body.text)
        if hit is not None:
            return {"answer": FAQ[hit], "used_llm": False}     # never reaches the GPU

        request = ChatCompletionRequest(
            model="qwen-0.5b",
            messages=[{"role": "user", "content": body.text}],
            max_tokens=96,
        )
        # stream is False, so the engine yields exactly ONE response.
        async for response in self.llm.chat.remote(request, None):
            return {"answer": response.choices[0].message.content, "used_llm": True}

# A GPU LLM deployment and a CPU deployment, same call, same position:
app = Assistant.bind(Triage.bind(), build_llm_deployment(llm_config))
```

Note: `.options(stream=True)` is required because `LLMServer.chat` is an async generator, and the second argument carries the original HTTP request when there is one. Set `stream=True` on the request instead and it yields many chunks, at which point `return` would keep only the first.

## 7. Deploying a production-ready LLM

The architecture does not change with the silicon, only the config does: same ingress-and-replicas shape, a bigger box.

<img src="https://anyscale-public-materials.s3.us-west-2.amazonaws.com/ray-serve-distributed-inference/diagrams/same_shape_bigger_box.png" loading="lazy" width="1000">

A production deployment is the Tier-1 app with a bigger model, a bigger accelerator, a parallelism dial set, and prefix caching on, shipped through the same builder.

```python
# Tier-2 production LLMConfig: a large dense model sharded across a full H100 node.
prod_config = LLMConfig(
    model_loading_config=dict(model_id="llama-3.1-70b", model_source="meta-llama/Llama-3.1-70B-Instruct"),
    accelerator_type="H100",
    deployment_config=dict(autoscaling_config=dict(min_replicas=1, max_replicas=4)),
    engine_kwargs=dict(
        tensor_parallel_size=8,        # shard the 70B across 8 GPUs of a node (section 4)
        max_model_len=131072,          # long context
        enable_prefix_caching=True,    # reuse shared-prefix KV across requests
    ),
)
```

```yaml
# service_prod.yaml: the SAME builder, deployed as a Service.
applications:
  - name: llm-serving-prod
    import_path: ray.serve.llm:build_openai_app    # the SAME import_path as Tier-1
    args:
      llm_configs:
        - model_loading_config:
            model_id: llama-3.1-70b
            model_source: meta-llama/Llama-3.1-70B-Instruct
          accelerator_type: H100
          engine_kwargs:
            tensor_parallel_size: 8
            max_model_len: 131072
            enable_prefix_caching: true
```

Note: an OpenAI-compatible ingress in front of `LLMServer` replicas, parallelism inside each replica (section 4), autoscaling across them (section 5). The architecture you learned in section 3 does not change; only the config does.

**Reference implementations**, in `code/llm/`:

- `serving/app.py` with `service.yaml`: sections 3 to 5. `cd code/llm/serving && serve run app:build_app`
- `composition/app.py` with `service.yaml`: section 6, plus a streaming endpoint
- `serving/service_prod.yaml`: the section 7 Tier-2 shape
- `serving/query.py`: client helpers